## Summit Healthcare
# Schema build outline

SHEDW Pharmacy Analytics Data Warehouse

 1. Data Warehouse Architecture
 2. Create Schemas
 3. Create Dimension Tables
 4. Create Fact Tables
 5. Verify the Database



## Database Architecture

Schemas
1. warehouse - stores dimension (DIM) and fact (FACT) tables
2. analytics - stores reporting views intended for Power BI

This warehouse follows a star schema design. FACT tables describe pharmacy events and DIM tables describe the people, medications, and departments involved in the events.

## Step 1. Create Schemas
Objective - organize database objects into logical groups.

In [ ]:
CREATE SCHEMA warehouse;
CREATE SCHEMA analytics;

\dn

## Step 2. Create dimension (DIM) tables.
Dimension tables describe the business entities used throughout SHEDW.
These tables rarely change and provide descriptive reporting information. 

In [ ]:
CREATE TABLE warehouse.dim_patient (
    pat_id INTEGER PRIMARY KEY,
    pat_mrn_id VARCHAR(20) UNIQUE NOT NULL,
    first_name VARCHAR(50) NOT NULL,
    last_name VARCHAR(50) NOT NULL,
    dob DATE,
    sex VARCHAR(20),
    city VARCHAR(50),
    state CHAR(2)
);

CREATE TABLE warehouse.dim_provider (
    provider_id INTEGER PRIMARY KEY,
    provider_name VARCHAR(100) NOT NULL,
    specialty VARCHAR(100)
);

CREATE TABLE warehouse.dim_department (
    department_id INTEGER PRIMARY KEY,
    department_name VARCHAR(100) NOT NULL,
    facility_id INTEGER,
    facility_name VARCHAR(100),
    unit_abbreviation VARCHAR(20)
);

CREATE TABLE warehouse.dim_medication (
    medication_id INTEGER PRIMARY KEY,
    generic_name VARCHAR(100) NOT NULL,
    therapeutic_class VARCHAR(100),
    formulary_status VARCHAR(20),
    controlled_substance BOOLEAN,
    control_level VARCHAR(10)
);

\dt warehouse.*

## Step 3. Create FACT tables.
Fact tables record mesaureable pharmacy events.
Each FACT table references one or more DIM table using foreign keys.

In [ ]:
CREATE TABLE warehouse.fact_medication_orders (
    order_id INTEGER PRIMARY KEY,

    patient_id INTEGER NOT NULL,
    provider_id INTEGER NOT NULL,
    department_id INTEGER NOT NULL,
    medication_id INTEGER NOT NULL,

    order_date TIMESTAMP,
    ordered_dose NUMERIC(10,2),
    ordered_route VARCHAR(30),
    priority VARCHAR(20),
    status VARCHAR(30),

    CONSTRAINT fk_patient
        FOREIGN KEY (patient_id)
        REFERENCES warehouse.dim_patient(pat_id),

    CONSTRAINT fk_provider
        FOREIGN KEY (provider_id)
        REFERENCES warehouse.dim_provider(provider_id),

    CONSTRAINT fk_department
        FOREIGN KEY (department_id)
        REFERENCES warehouse.dim_department(department_id),

    CONSTRAINT fk_medication
        FOREIGN KEY (medication_id)
        REFERENCES warehouse.dim_medication(medication_id)
);

CREATE TABLE warehouse.fact_medication_verification (

    order_id INTEGER PRIMARY KEY,

    verifying_provider_id INTEGER,
    verify_time TIMESTAMP,
    queue_time INTEGER,
    stat_order BOOLEAN,

    CONSTRAINT fk_order_verify
        FOREIGN KEY (order_id)
        REFERENCES warehouse.fact_medication_orders(order_id),

    CONSTRAINT fk_verify_provider
        FOREIGN KEY (verifying_provider_id)
        REFERENCES warehouse.dim_provider(provider_id)
);

CREATE TABLE warehouse.fact_medication_dispense (

    order_id INTEGER PRIMARY KEY,

    dispense_time TIMESTAMP,
    dispense_quantity NUMERIC(10,2),
    dispense_quantity_unit VARCHAR(20),
    return_status BOOLEAN,

    CONSTRAINT fk_order_dispense
        FOREIGN KEY (order_id)
        REFERENCES warehouse.fact_medication_orders(order_id)
);

CREATE TABLE warehouse.fact_medication_administration (

    order_id INTEGER PRIMARY KEY,

    admin_department INTEGER,
    admin_time TIMESTAMP,
    missed_dose BOOLEAN,
    admin_provider_id INTEGER,

    CONSTRAINT fk_order_admin
        FOREIGN KEY (order_id)
        REFERENCES warehouse.fact_medication_orders(order_id),

    CONSTRAINT fk_admin_department
        FOREIGN KEY (admin_department)
        REFERENCES warehouse.dim_department(department_id),

    CONSTRAINT fk_admin_provider
        FOREIGN KEY (admin_provider_id)
        REFERENCES warehouse.dim_provider(provider_id)
);

 \dt warehouse.*

## Step 4. Verify the Database.
The following command line commands verify the table creation codes executed successfully.

In [ ]:
-- List schemas
\dn

-- List tables
\dt warehouse.*

-- Describe the medication orders table
\d warehouse.fact_medication_orders

## Step 5. Next Steps.

The next phase of this project will include:
    - Generating realistic (sim) sample data
    - Loading data into the SHEDW
    - Writing analytical SQL queries
    - Building Power BI dashboards
    - Migrating database to Azure cloud platform